# Matching products

**Abt-Buy and Amazon-Google: two public benchmarks, complete ground truth, and one finding that reorders the whole lane.**

Two retailers list the same camera case. One calls it `Canon Deluxe Black Digital Camera Case - 2595B002`, the other `Canon PSC-85 Soft Camera Case - 2595B002`. Nothing about those titles matches except eight characters in the middle.

This notebook runs arche's experimental electronics lane against the [Leipzig product benchmarks](https://dbs.uni-leipzig.de/research/projects/benchmark-datasets-for-entity-resolution) — CC-BY-4.0, and their mappings are *complete*, so false merges are visible rather than assumed.

**Three things it establishes.**

1. A plain name matcher gets **F1 0.3443** here. Product titles are marketing copy.
2. The identity is a rare code, and **rarity — not the code's shape — is the signal.**
3. The lane barely helps on general merchandise, which is why it ships named `product_electronics` rather than `product`.


## Setup


In [1]:
import csv, sys
from collections import Counter, defaultdict
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "packages" / "arche-core").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "packages" / "arche-core" / "src"))

from arche.resolve import crosswalk, ENTITY_PACKS
from arche.resolve._productcode import (
    build_code_table, code_rarity, extract_product_code_candidates,
)

DATA = REPO / "data" / "er_bench" / "products"


def read(name):
    with open(DATA / name, encoding="utf-8-sig", errors="replace", newline="") as fh:
        return list(csv.DictReader(fh))


abt, buy = read("Abt.csv"), read("Buy.csv")
truth = {(r["idAbt"], r["idBuy"]) for r in read("abt_buy_perfectMapping.csv")}
print(f"Abt {len(abt)} x Buy {len(buy)}  ->  {len(truth)} true pairs")
print(f"possible pairs: {len(abt) * len(buy):,}   positive rate: "
      f"{100 * len(truth) / (len(abt) * len(buy)):.4f}%")


Abt 1081 x Buy 1092  ->  1097 true pairs
possible pairs: 1,180,452   positive rate: 0.0929%


## 1. What a product title actually looks like

Read five true pairs before writing any matcher.


In [2]:
ia = {r["id"]: r for r in abt}
ib = {r["id"]: r for r in buy}
for a, b in list(truth)[:5]:
    if a in ia and b in ib:
        print(f"  ABT  {ia[a]['name'][:74]}")
        print(f"  BUY  {ib[b]['name'][:74]}")
        print()


  ABT  Transcend 8GB Micro Secure Digital Memory Card - TS8GUSDHC6
  BUY  Transcend 8GB microSDHC Card - Class 6 - TS8GUSDHC6

  ABT  Escort Passport Radar And Laser Detector - Black Finish - 8500
  BUY  Escort X50 Passport 8500 Radar Detector - High Intensity Red Display - 850

  ABT  LG 24' LDF6920ST Fully Integrated Built In Stainless Steel Dishwasher - LD
  BUY  LG LDF6920ST Fully Integrated Dishwasher with 5 Wash Cycles, 3 Spray Arms,

  ABT  Logitech Black V220 Cordless Optical Mouse For Notebooks - 910000153
  BUY  Logitech V220 Cordless Optical Mouse for Notebooks - 910-000153

  ABT  Samsung 19' Black Flat Panel Series 6 LCD HDTV - LN19A650
  BUY  Samsung 6 Series LN19A650 19' LCD TV



The pattern is the same every time: a manufacturer code, wrapped in whatever copy each retailer felt like writing. `Deluxe Black` against `PSC-85 Soft`. `Super Capacity Drum` against nothing at all.

The code is the identity. Everything else is noise that happens to be longer.

## 2. The baseline, so the improvement is measurable


In [3]:
A = [{"id": r["id"], "name": r["name"]} for r in abt]
B = [{"id": r["id"], "name": r["name"]} for r in buy]


def score(comparators=None, entity=None, label=""):
    kw = {"comparators": comparators} if comparators else {"entity": entity}
    res = crosswalk(A, B, id_field="id", **kw)
    pred = {(e["a_id"], e["b_id"]): e for e in res["matches"]}
    tp = sum(1 for k, e in pred.items() if e["decision"] == "match" and k in truth)
    fp = sum(1 for k, e in pred.items() if e["decision"] == "match" and k not in truth)
    rv = sum(1 for k, e in pred.items() if e["decision"] == "review" and k in truth)
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / len(truth)
    print(f"  {label:<26} P={p:.4f}  R={r:.4f}  F1={2*p*r/(p+r) if p+r else 0:.4f}  "
          f"surfaced={(tp+rv)/len(truth):.4f}  (TP {tp}, FP {fp})")
    return p, r


score(comparators=[{"field": "name", "kind": "name", "weight": 2.0},
                   {"field": "name", "kind": "tftoken", "weight": 2.0}],
      label="name only (baseline)")


  name only (baseline)       P=0.7954  R=0.2197  F1=0.3443  surfaced=0.6217  (TP 241, FP 62)


(0.7953795379537953, 0.21969006381039197)

**F1 0.3443.** A name matcher is close to useless here, and that is not a bug in the comparator — it is a fact about product titles. The strings genuinely differ.

## 3. The signal is rarity, not shape

The obvious move is a regex for model numbers. Here is why that alone is not enough.

`extract_product_code_candidates` deliberately returns *candidates* — anything that could be a manufacturer code, a retailer SKU, a spec or a quantity. A regex cannot tell those apart. Rarity can.


In [4]:
for title in ("Fellowes Powershred Personal SB-97Cs Confetti Cut Shredder - 3219701",
              "Sony 1080p 16GB Handycam HDRCX150"):
    print(f"  {title[:62]:64} -> {sorted(extract_product_code_candidates(title))}")


  Fellowes Powershred Personal SB-97Cs Confetti Cut Shredder - 3   -> ['3219701', 'sb97cs']
  Sony 1080p 16GB Handycam HDRCX150                                -> ['16gb', 'hdrcx150']


Note `3219701` — a retailer SKU, not a manufacturer code — and `16gb`, which is a specification. Both look exactly like model numbers to a regex.

Now block on a shared code and condition on how rare that code is.


In [5]:
MA = {r["id"]: extract_product_code_candidates(r["name"]) for r in abt}
MB = {r["id"]: extract_product_code_candidates(r["name"]) for r in buy}
tf = build_code_table([r["name"] for r in abt] + [r["name"] for r in buy])

index = defaultdict(list)
for r in buy:
    for code_ in MB[r["id"]]:
        index[code_].append(r["id"])

candidates = defaultdict(set)
for r in abt:
    for code_ in MA[r["id"]]:
        for other in index[code_]:
            candidates[(r["id"], other)].add(code_)

hit = sum(1 for k in candidates if k in truth)
print(f"pairs sharing any code: {len(candidates)}   true: {hit}   "
      f"precision {hit/len(candidates):.4f}")
print()
print(f"{'rarest shared code, doc freq':<30} {'pairs':>7} {'true':>6} {'precision':>10}")
for lo, hi in ((1, 2), (3, 4), (5, 9), (10, 19), (20, 10**6)):
    sel = [k for k, cs in candidates.items()
           if lo <= min(tf._as_counts().get(c, 0) for c in cs) <= hi]
    if not sel:
        continue
    t = sum(1 for k in sel if k in truth)
    label = f"{lo}-{hi if hi < 10**6 else '+'}"
    print(f"{label:<30} {len(sel):>7} {t:>6} {t/len(sel):>10.4f}")


pairs sharing any code: 881   true: 781   precision 0.8865

rarest shared code, doc freq     pairs   true  precision
1-2                                754    752     0.9973
3-4                                 47     23     0.4894
5-9                                 55      6     0.1091
10-19                               25      0     0.0000


**That table is the whole lane.**

A code seen once or twice identifies almost perfectly. A code seen twenty or more times — `1080p`, `16gb`, `720p` — identifies **nothing at all**: hundreds of candidate pairs, not one true match among them.

So the signal was never "looks like a model number". It is "is rare". `1080p` is the `General Hospital` of consumer electronics, and the fix is the frequency table arche already uses for places and people, not a cleverer regex.


In [6]:
print(f"{'code':<12} {'doc freq':>9} {'rarity':>8}")
for c in ("2595b002", "feq332wh", "sb97cs", "16gb"):
    print(f"  {c:<12} {tf._as_counts().get(c, 0):>7.0f} {code_rarity(c, tf):>8.3f}")
print()
print("DISTINCTIVE_FLOOR = 0.75 — only the first three can clear the gate unaided.")


code          doc freq   rarity
  2595b002           3    0.667
  feq332wh           2    1.000
  sb97cs             2    1.000
  16gb              11    0.182

DISTINCTIVE_FLOOR = 0.75 — only the first three can clear the gate unaided.


## 4. The shipped lane


In [7]:
score(entity="product_electronics", label="product_electronics")
print()
for spec in ENTITY_PACKS["product_electronics"]:
    print(" ", {k: v for k, v in spec.items() if k != "category"})


  product_electronics        P=0.9707  R=0.6636  F1=0.7883  surfaced=0.7521  (TP 728, FP 22)

  {'field': 'name', 'kind': 'name', 'weight': 1.5}
  {'field': 'name', 'kind': 'code', 'weight': 3.0}
  {'field': 'name', 'kind': 'tftoken', 'weight': 1.5}
  {'field': 'name', 'kind': 'spec', 'weight': 0.5, 'refutes_below': 0.5}


**F1 0.3443 → 0.7883**, precision 0.9707, 22 false merges.

Two details in that pack worth reading.

`code` carries the highest weight because a shared *rare* code is the identity, but it returns **0.0 rather than a veto** when both sides have codes and share none — 18.6% of true pairs are in that position, because accessories, bundles and retailer SKUs legitimately disagree. A hard conflict rule would refute all of them.

`spec` uses `refutes_below` under a **purchasable-variant identity contract**: a 16GB and a 32GB player are different products however alike their titles. On this corpus it is exactly neutral — it earns its place from the contract, not from the benchmark, and the changelog says so.

## 5. Where it stops working


In [8]:
amz, goo = read("Amazon.csv"), read("GoogleProducts.csv")
t2 = {(r["idAmazon"], r["idGoogleBase"]) for r in read("Amzon_GoogleProducts_perfectMapping.csv")}
A2 = [{"id": r["id"], "name": r["title"]} for r in amz]
B2 = [{"id": r["id"], "name": r["name"]} for r in goo]


def score2(kw, label):
    res = crosswalk(A2, B2, id_field="id", **kw)
    pred = {(e["a_id"], e["b_id"]): e for e in res["matches"]}
    tp = sum(1 for k, e in pred.items() if e["decision"] == "match" and k in t2)
    fp = sum(1 for k, e in pred.items() if e["decision"] == "match" and k not in t2)
    p, r = tp / (tp + fp), tp / len(t2)
    print(f"  {label:<26} P={p:.4f}  R={r:.4f}  F1={2*p*r/(p+r):.4f}  (TP {tp}, FP {fp})")


score2({"comparators": [{"field": "name", "kind": "name", "weight": 2.0},
                        {"field": "name", "kind": "tftoken", "weight": 2.0}]},
       "name only (baseline)")
score2({"entity": "product_electronics"}, "product_electronics")


  name only (baseline)       P=0.4898  R=0.3338  F1=0.3971  (TP 434, FP 452)


  product_electronics        P=0.4863  R=0.3408  F1=0.4007  (TP 443, FP 468)


On Amazon-GoogleProducts — general merchandise rather than consumer electronics — the lane moves F1 from 0.3971 to 0.4007 and **precision falls**, 0.4898 to 0.4863. That is +9 true matches bought with +16 false ones: a marginal precision of **0.36** on the pairs it changes.

The F1 gain is real and it is not worth having. Reporting only F1 would have hidden that.

This is why the pack is named `product_electronics`, is flagged `experimental=True`, and why a test asserts no generic `product` pack exists. The rules that work here fail elsewhere by construction: Levi's `501` is rejected twice by thresholds that exist to filter prices and years, `32x32` looks like a model and is not, and reading `600mg` as a drug's model code would be dangerous.

Adding food, books or apparel is a **category registration plus a benchmark**, not a change to any comparator:

```python
from arche.resolve._productcode import ProductCategory, register_category

register_category(ProductCategory(
    name="apparel",
    min_code_len=3, min_bare_number_len=3,   # Levi's 501 is a real model
    identity_units=("inch",),
    stop_codes=frozenset({"32x32"}),
))
```

## What this establishes, and what it does not

**Establishes.** On a public benchmark with complete ground truth, neither built nor labelled by us, the lane takes F1 from 0.3443 to 0.7883 at precision 0.9707. The mechanism is rarity, measured, not asserted.

**Does not establish.**

* **One vertical.** Consumer electronics. The Amazon-Google result above is the evidence that it does not generalise as-is.
* **The stop list does nothing here.** With `stop_codes` emptied the end-to-end result is byte-identical — the frequency table already suppresses `1080p`. The list earns its place on catalogues too small to estimate frequency from, which this benchmark cannot show.
* **The `spec` refutation is unvalidated.** Exactly neutral on Abt-Buy, entirely inert on Amazon-Google, where no true pair carries a comparable unit. 47 of 1,097 pairs is not an evidence base.
* **Titles only.** The pack reads `name`. `description`, `manufacturer` and `price` are all present in the data and all unused.

*Related: [the product tutorial](../../docs-site/docs/tutorials/products.md) · [what is the false-merge rate?](06_what_is_the_false_merge_rate.ipynb) · provenance in `data/er_bench/SOURCES.md`*
